# <font color='CB6D51'>Qblox Sequence Templates</font>

This file contains templates for Qblox Q1ASM sequences and related Pyhton code for a number of situations.\
The templates are organized by module, then by situation.

Created: 9 Dec 2024\
Last updated: 9 Dec 2024

Tested: 9 Dec 2024\
On firmware: 0.9.2\
On qblox_instruments: 0.14.2

Author: Kyle MacRobbie

---

## Setup

In [ ]:
# Imports, just so that these code blocks don't come up with errors
import sequence_helper as sh
import math
import numpy as np
from qblox_instruments.qcodes_drivers.module import Module
from qblox_instruments import Cluster
import matplotlib.pyplot as plt
qcm_module = Module
qrm_module = Module
rf_module = Module
cluster = Cluster

---

## <font color='#7FFFD4'>QCM templates</font>

### Playing waveforms from one output (less than 16 µs)

In [ ]:
# Q1ASM sequence
sequence = f""" 
	wait_sync		4			# Sync with other sequencers
	play			0,0,1000	# Play waveform 0
	wait			1000		# Wait 1000 ns
	play			1,1,1000	# Play waveform 1
	stop
"""

In [ ]:
# Python code
amplitude = 2					# V
amplitude_q1 = amplitude / 2.5	# Value that gets put into the sequence, accounting for the QCM's output range
waveform_length = 1000			# ns

waveforms = {
	"block" : {
		"data": [amplitude_q1 for i in range(waveform_length)],
		"index": 0
	},
	"sine": {
		"data": [amplitude_q1 * math.sin((2 * math.pi / waveform_length) * i) for i in range(0, waveform_length)],
		"index": 1
	},
}

sequence_dict = {
	"waveforms": waveforms,
	"weights": {},
	"acquisitions": {},
	"program": sequence
}

qcm_module.sequencer0.sequence(sequence_dict)	# Upload sequence
qcm_module.disconnect_outputs()					# Disconnect outputs
qcm_module.sequencer0.connect_out0('I')			# Connect to input 0 on path I
qcm_module.sequencer0.sync_en(True)				# Enable sync to other sequencers
qcm_module.arm_sequencer(0)						# Arm sequencer 0

### Playing gate voltages out of one output (more than 16 µs)

In [ ]:
block_voltage = 1										# V
block_voltage_q1 = round((block_voltage / 2.5) * 32767)	# Value that gets put into the sequence, accounting for the QCM's output range
block_duration = 10000									# ns
ramp_duration = 10000									# ns

ramp_start = 1											# V
ramp_end = 0											# V
step_duration = 500										# ns
num_ramp_steps = int(ramp_duration/step_duration)		# Convert to Q1ASM value
step_size = (ramp_end - ramp_start) / num_ramp_steps	# Voltage jump on each step
step_size_q1 = round((step_size / 2.5) * 32767)			# Convert to Q1ASM value
ramp_start_q1 = round((ramp_start / 2.5) * 32767)		# Convert to Q1ASM value

sequence = f""" 
		move	{num_ramp_steps},R0 	# Ramp iterator
		move	{ramp_start_q1},R1		# Ramp value

		wait_sync		4				# Sync with other sequencers

		set_awg_offs	{block_voltage_q1},{block_voltage_q1}	# Set offset to the block voltage
		upd_param		{block_duration}						# Apply offset and wait

	
	ramp_loop:
		set_awg_offs	R1,R1					# Set offset to the currect step voltage
		upd_param		{step_duration}			# Apply offset and wait
		sub				R1,{-step_size_q1},R1	# Update the step voltage
		loop			R0,@ramp_loop			# Loop


		set_awg_offs	0,0		# Return the voltage offset to 0 before stopping
		upd_param		4		# Apply 0 offset

		stop
"""

In [ ]:
# Python code
sequence_dict = {
	"waveforms": {},
	"weights": {},
	"acquisitions": {},
	"program": sequence
}

qcm_module.sequencer0.sequence(sequence_dict)	# Upload sequence
qcm_module.disconnect_outputs()					# Disconnect any current outputs
qcm_module.sequencer0.connect_out0('I')			# Connect to output 0 on path I
qcm_module.sequencer0.sync_en(True)				# Enable sync with other sequencers
qcm_module.arm_sequencer(0)						# Arm sequencer 0

---

## <font color='#7FFFD4'>QRM templates</font>

### Playing waveforms from one output (less than 16 µs)

In [ ]:
# Q1ASM sequence
sequence = f""" 
	wait_sync		4			# Sync with other sequencers
	play			0,0,1000	# Play waveform 0
	wait			1000		# Wait 1000 ns
	play			1,1,1000	# Play waveform 1
	stop
"""

In [ ]:
# Python code
amplitude = 2					# V
amplitude_q1 = amplitude / 0.5	# Value that gets put into the sequence, accounting for the QCM's output range
waveform_length = 1000			# ns

waveforms = {
	"block" : {
		"data": [amplitude_q1 for i in range(waveform_length)],
		"index": 0
	},
	"sine": {
		"data": [amplitude_q1 * math.sin((2 * math.pi / waveform_length) * i) for i in range(0, waveform_length)],
		"index": 1
	},
}

sequence_dict = {
	"waveforms": waveforms,
	"weights": {},
	"acquisitions": {},
	"program": sequence
}

qrm_module.sequencer0.sequence(sequence_dict)	# Upload sequence
qrm_module.disconnect_outputs()					# Disconnect outputs
qrm_module.sequencer0.connect_out0('I')			# Connect to output 0 through path I
qrm_module.sequencer0.sync_en(True)				# Enable sync
qrm_module.arm_sequencer(0)						# Arm sequencer 0

### Playing gate voltages out of one output (more than 16 µs)

In [ ]:
block_voltage = 1										# V
block_voltage_q1 = round((block_voltage / 0.5) * 32767)	# Converting to the Q1ASM value
block_duration = 10000									# ns
ramp_duration = 10000									# ns

ramp_start = 1		# Starting voltage of the ramp
ramp_end = 0		# Ending voltage of the ramp
step_duration = 500	# ns

num_ramp_steps = int(ramp_duration/step_duration)		# Number of steps the ramp is being broken in to
step_size = (ramp_end - ramp_start) / num_ramp_steps	# Voltage change of each step
step_size_q1 = round((step_size / 0.5) * 32767)			# Converting to the Q1ASM value
ramp_start_q1 = round((ramp_start / 0.5) * 32767)		# Converting to the Q1ASM value

sequence = f""" 
		move	{num_ramp_steps},R0 	# ramp iterator
		move	{ramp_start_q1},R1		# ramp value
		wait_sync		4				# Sync with other sequencers

		set_awg_offs	{block_voltage_q1},{block_voltage_q1}	# Set voltage offset to the block voltage
		upd_param		{block_duration}						# Apply offset and wait

	
	ramp_loop:
		set_awg_offs	R1,R1					# Set voltage offset to the current step
		upd_param		{step_duration}			# Apply offset and wait
		sub				R1,{-step_size_q1},R1	# Step the voltage
		loop			R0,@ramp_loop			# Loop


		set_awg_offs	0,0		# Reset the voltage offset to 0 before stopping
		upd_param		4		# Apply the 0 offset

		stop
"""

In [ ]:
# Python code
sequence_dict = {
	"waveforms": {},
	"weights": {},
	"acquisitions": {},
	"program": sequence
}

qrm_module.sequencer0.sequence(sequence_dict)	# Upload sequence
qrm_module.disconnect_outputs()					# Disconnect outputs
qrm_module.sequencer0.connect_out0('I')			# Connect to output 0 through path I
qrm_module.sequencer0.sync_en(True)				# Enable sync with other sequencers
qrm_module.arm_sequencer(0)						# Arm sequencer 0

### Acquire through one input (less than 16 µs)

In [ ]:
# Q1ASM sequence
sequence = f""" 
	wait_sync		4			# Sync with other sequencers
	acquire			0,0,1000	# Acquire to acquisition 0
	stop
"""

In [ ]:
# Python code
acquisitions = {
	"acq": {"num_bins": 1, "index": 0}
}

sequence_dict = {
	"waveforms": {},
	"weights": {},
	"acquisitions": acquisitions,
	"program": sequence
}

qrm_module.sequencer0.sequence(sequence_dict)				# Upload sequence
qrm_module.disconnect_outputs()								# Disconnect outputs
qrm_module.disconnect_inputs()								# Disconnect inputs
qrm_module.sequencer0.connect_acq_I("in0") 					# Connect to input 0 through path I
qrm_module.scope_acq_sequencer_select(0)					# Select sequencer 0 for scope mode 
qrm_module.scope_acq_trigger_mode_path0('sequencer')		# Trigger the acquisition with the sequence
qrm_module.sequencer0.delete_acquisition_data(all = True)	# Delete previous acquisition
qrm_module.sequencer0.integration_length_acq(1000)			# Set integration length
qrm_module.sequencer0.sync_en(True)							# Enable sync
qrm_module.arm_sequencer(0)									# Arm sequencer 0

In [ ]:
# Plotting
qrm_module.get_acquisition_status(0) 			# Wait for the sequencer to stop with a timeout period of one minute.
qrm_module.store_scope_acquisition(0, 'acq') 	# Move acquisition data from temporary memory to acquisition list.
readout_data = qrm_module.get_acquisitions(0) 	# Get acquisition list from instrument.
data0 = readout_data['acq']['acquisition']['scope']['path0']['data']
data1 = readout_data['acq']['acquisition']['scope']['path1']['data']
fig, ax = plt.subplots(1, 1, figsize = (14, 4))	# 
ax.plot(data0, alpha = 0.9, label = "Path 0")
ax.plot(data1, alpha = 0.9, label = "Path 1")
ax.set_title("QRM input")
ax.set_xlabel("Time [ns]")
ax.set_ylabel("Input [V]")
ax.grid(ls = '--')
plt.legend()
plt.show()

### Acquire through one input (more than 16 µs)

In [ ]:
# Q1ASM sequence
duration = 1e6
resolution = 500
num_bins = math.ceil(duration / resolution)

sequence = f"""
		move 		0,R0
		move		{num_bins},R1
		wait_sync	4
		wait		150

	loop:
		acquire		0,R0,{resolution}
		add			R0,1,R0
		loop		R1,@loop

		stop
"""

In [ ]:
# Python code
sequence = {
	"waveforms": {},
	"weights": {},
	"acquisitions": acquisitions,
	"program": sequence,
}

acquisitions = {
	f"{input[0]}": {"num_bins": num_bins, "index": 0}
}

qrm_module.sequencer0.sequence(sequence_dict)
qrm_module.disconnect_outputs()
qrm_module.disconnect_inputs()
qrm_module.sequencer0.connect_acq_I("in0") 
qrm_module.scope_acq_sequencer_select(0)
qrm_module.scope_acq_trigger_mode_path0('sequencer')
qrm_module.sequencer0.delete_acquisition_data(all = True)
qrm_module.sequencer0.integration_length_acq(resolution)
qrm_module.sequencer0.sync_en(True)
qrm_module.arm_sequencer(0)

In [ ]:
# Plotting
num_bins = len(readout_data['acq']['acquisition']['bins']['integration']['path0'])
resolution = 250
data0 = np.array(readout_data['acq']['acquisition']['bins']['integration']['path0']) / resolution
data1 = np.array(readout_data['acq']['acquisition']['bins']['integration']['path1']) / resolution
t = np.arange(resolution / 2, resolution * num_bins + 0.1, resolution)
fig, ax = plt.subplots(1, 1, figsize = (14, 4))
ax.plot(t, data0, alpha = 0.9, label = "Path 0")
ax.plot(t, data1, alpha = 0.9, label = "Path 1")
ax.set_title("QRM input")
ax.set_xlabel("Time [ns]")
ax.set_ylabel("Input [V]")
ax.grid(ls = '--')
plt.legend()
plt.show()

---

## <font color='#7FFFD4'>QCM-RF templates</font>

### Playing a single RF pulse

In [ ]:
# Q1ASM sequence
sequence = f""" 
	wait_sync		4
	set_mrk			{0b1001}
	play			0,0,1000
	set_mrk			{0b0000}
	upd_param		4
	stop
"""

In [ ]:
# Python code
amplitude = 0.499					# Volts
amplitude_q1 = amplitude / 0.5	# Value that gets put into the sequence, accounting for the QCM's output range
waveform_length = 2000

waveforms = {
	"block" : {
		"data": [amplitude_q1 for i in range(waveform_length)],
		"index": 0
	}
}

sequence_dict = {
	"waveforms": waveforms,
	"weights": {},
	"acquisitions": {},
	"program": sequence
}

rf_module.sequencer0.sequence(sequence_dict)
rf_module.disconnect_outputs()
rf_module.sequencer0.connect_out0(True)
rf_module.sequencer0.mod_en_awg(True)
rf_module.out0_lo_en(True)
rf_module.sequencer0.nco_freq(50e6)
rf_module.out0_lo_freq(2e9)
rf_module.sequencer0.sync_en(True)
rf_module.arm_sequencer(0)

---

## <font color='#7FFFD4'>Stopping and resetting template</font>

In [ ]:
cluster.stop_sequencer()

print("QRM sequencer 0: " + str(qrm_module.get_sequencer_status(0)))
print("QRM sequencer 1: " + str(qrm_module.get_sequencer_status(1)))
print("QCM sequencer 0: " + str(qrm_module.get_sequencer_status(0)))
print("QCM sequencer 1: " + str(qrm_module.get_sequencer_status(1)))
print("RF sequencer 0:  " + str(qrm_module.get_sequencer_status(0)))
print("RF sequencer 1:  " + str(qrm_module.get_sequencer_status(1)))

# Reset the cluster
cluster.reset()
print(cluster.get_system_status())